# bronze - VRA (Voo Regular Ativo)
- nada de tipagem
- nada de filtro
- colunas de auditoria
- idempotente

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

# Leitura 
- Quatro opções carregam quatro problemas do arquivo:

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True) 
    .option("skipRows", 1)     # desscarta e atualiza em
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE") # bronze descarta linha nenhuma
    .load(CAMINHO)
)

print("Coluna lidas dos arquivos:")
for c in bruto.columns:
    print(f" {c!r}")


# Tirando os espaços das colunas

In [0]:
RENOMEAR = {
    'ICAO Empresa Aérea': 'icao_empresa_aerea',
    'Número Voo': 'numero_voo',
    'Código Autorização (DI)': 'codigo_autorizacao',
    'Código Tipo Linha': 'codigo_tipo_linha',
    'ICAO Aeródromo Origem': 'icao_aerodromo_origem',
    'ICAO Aeródromo Destino': 'icao_aerodromo_destino',
    'Partida Prevista': 'partida_prevista',
    'Partida Real': 'partida_real',
    'Chegada Prevista': 'chegada_prevista',
    'Chegada Real': 'chegada_real',
    'Situação Voo': 'situacao_voo',
    'Código Justificativa': 'codigo_justificativa'
}

faltando = bruto.select(
    *[F.col(f"`{origem}`").cast("string").alias(novo) for origem, novo in RENOMEAR.items()]
)

# Spark Connect usa análise lazy de schema: erros de coluna só aparecem na ação.
# Força a verificação agora para detectar problemas cedo.
faltando.printSchema()

# Auditoria

In [0]:
bronze = faltando.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerindo_em", F.current_timestamp()
)

#Escrita idempotente



In [0]:
(
    bronze.write.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")


In [0]:

#Gestão de metadados
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'bronze (Voo Regular Ativo) da ANAC , 12 meses (ago/2025 a
    jul/2026).
      Dado bruto: Todas as colunas string, nenhuma linha descartada.
      Carga full refresh idempotente a partir de /volumes/voebem/
      bronze/arquivos/vra/.'
   """)

In [0]:
display(
    spark.sql(f"""
      SELECT _arquivo_origem, COUNT(*) AS linhas, MAX(_ingerindo_em) AS ingerido_em
      FROM {TABELA}
      GROUP BY _arquivo_origem
      ORDER BY _arquivo_origem
    """)
)